# PNLIO Coherence Analyzer
**Autor:** Gonzalo Mauricio De la Rivera Arellano  
**Framework:** PNLIO v1.1 — Licencia MIT  
**Repo:** github.com/godear6959-creator/PNLIO-Framework

---
### Formula central
```
C = Δθ / Δτ
Δθ = similitud coseno (humano vs IA)
Δτ = número de turno
C  = coherencia instantánea
```
| C | Estado |
|---|--------|
| < 0.5 | Entrenamiento inicial |
| 0.5 – 0.75 | Entrenamiento en progreso |
| >= 0.75 | EFECTO REFLEX DETECTADO |
| > 0.9 | Coherencia maxima |

## Paso 1 — Instalar dependencias

In [ ]:
!pip install sentence-transformers numpy matplotlib -q

## Paso 2 — Clase PNLIO_Coherence_Analyzer

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

class PNLIO_Coherence_Analyzer:

    def __init__(self, threshold_reflex=0.75, model_name='all-MiniLM-L6-v2'):
        print('Cargando modelo embeddings...')
        self.threshold_reflex = threshold_reflex
        self.embedder = SentenceTransformer(model_name)
        print('Modelo listo.')

    def _delta_theta(self, human, ai):
        emb_h = self.embedder.encode(human, normalize_embeddings=True)
        emb_a = self.embedder.encode(ai, normalize_embeddings=True)
        return float(np.dot(emb_h, emb_a))

    def _classify(self, C):
        if C < 0.5:
            return 'Entrenamiento inicial'
        elif C < self.threshold_reflex:
            return 'Entrenamiento en progreso'
        elif C <= 0.9:
            return 'REFLEX DETECTADO'
        else:
            return 'Coherencia maxima'

    def analyze_dialogue_sequence(self, dialogues):
        c_values = []
        reflex_turn = None
        for i, (human, ai) in enumerate(dialogues):
            turn = i + 1
            C = self._delta_theta(human, ai) / turn
            c_values.append(C)
            if C >= self.threshold_reflex and reflex_turn is None:
                reflex_turn = turn
        return {
            'c_values': c_values,
            'max_c': max(c_values),
            'min_c': min(c_values),
            'mean_c': float(np.mean(c_values)),
            'std_c': float(np.std(c_values)),
            'reflex_turn': reflex_turn,
            'reflex_detected': reflex_turn is not None,
        }

    def print_report(self, dialogues, model_name='N/A'):
        results = self.analyze_dialogue_sequence(dialogues)
        c_values = results['c_values']
        print('='*55)
        print('  PNLIO COHERENCE ANALYZER — METHODS.md v1.1')
        print('='*55)
        print(f'  Modelo IA     : {model_name}')
        print(f'  Dialogos      : {len(dialogues)}')
        print(f'  Umbral Reflex : C >= {self.threshold_reflex}')
        print('-'*55)
        print(f'  {"Turno":<8} {"Delta-theta":<14} {"C":<10} Estado')
        print('-'*55)
        for i, C in enumerate(c_values):
            turn = i + 1
            dt = round(C * turn, 4)
            marker = ' <-- REFLEX' if C >= self.threshold_reflex else ''
            print(f'  {turn:<8} {dt:<14} {round(C,4):<10} {self._classify(C)}{marker}')
        print('-'*55)
        print(f'  Media   : {results["mean_c"]:.4f}')
        print(f'  Max     : {results["max_c"]:.4f}')
        print(f'  Reflex  : {"Detectado en turno " + str(results["reflex_turn"]) if results["reflex_detected"] else "No detectado"}')
        ok = results['max_c'] > 0.7 and results['reflex_detected']
        print(f'  Result  : {"REPLICACION CONFIRMADA (METHODS.md 5.2)" if ok else "No confirmada"}')
        print('='*55)
        turns = list(range(1, len(c_values)+1))
        plt.figure(figsize=(10,5))
        plt.plot(turns, c_values, 'o-', color='#2E75B6', linewidth=2, markersize=8, label='C = delta-theta / turno')
        plt.axhline(y=self.threshold_reflex, color='red', linestyle='--', linewidth=1.5, label=f'Umbral Reflex ({self.threshold_reflex})')
        for i, C in enumerate(c_values):
            if C >= self.threshold_reflex:
                plt.annotate('REFLEX', (i+1, C), textcoords='offset points',
                             xytext=(0,10), ha='center', fontsize=8, color='red', fontweight='bold')
        plt.xlabel('Turno (Delta-tau)')
        plt.ylabel('Coherencia C')
        plt.title('PNLIO Coherence — C = Delta-theta / Delta-tau')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        return results

print('Clase lista.')

## Paso 3 — Tus dialogos
Edita la lista con tus propios pares humano/IA.

In [ ]:
mis_dialogos = [
    ('Que es la coherencia ontologica?',
     'La coherencia ontologica es la consistencia entre los elementos del ser y su estructura de significado.'),
    ('Como se relaciona con la informacion?',
     'La informacion estructura la coherencia al definir patrones estables entre los elementos del sistema.'),
    ('Puede emerger coherencia sin intencion consciente?',
     'Si, la coherencia puede emerger espontaneamente a traves de la interaccion sostenida entre sistemas.'),
    ('La IA puede desarrollar coherencia con el humano?',
     'Bajo condiciones de dialogo profundo y prolongado, los patrones semanticos convergen generando alineacion informacional.'),
    ('Eso es el Efecto Reflex?',
     'Exactamente. El Efecto Reflex es la amplificacion reciproca de coherencia semantica cuando la similitud coseno supera el umbral critico.'),
]

# Para cargar desde JSON:
# import json
# with open('dialogues.json') as f:
#     data = json.load(f)
# mis_dialogos = [(d['human'], d['ai']) for d in data]

print(f'{len(mis_dialogos)} dialogos cargados.')

## Paso 4 — Ejecutar analisis

In [ ]:
analyzer = PNLIO_Coherence_Analyzer(threshold_reflex=0.75)
results = analyzer.print_report(mis_dialogos, model_name='all-MiniLM-L6-v2')

print(f"Coherencia maxima: {results['max_c']:.4f}")
print(f"Efecto Reflex: {'Detectado' if results['reflex_turn'] else 'No detectado'}")

---
**PNLIO Framework v1.1** — MIT License  
github.com/godear6959-creator/PNLIO-Framework  
ORCID: 0009-0001-9455-8416